# RAG Pipeline Evaluation

This notebook evaluates the deployed RAG pipeline against a small hand-labeled
question set (`qa_dataset.json`). It measures:

1. **Retrieval Hit-Rate@k** — did at least one retrieved chunk actually contain
   the expected keywords for the question?
2. **Mean Reciprocal Rank (MRR)** — how high up the reranked list was the first
   relevant chunk?
3. **Retrieval score breakdown** — BM25 vs vector vs rerank scores per question,
   to visually confirm hybrid retrieval + reranking is doing meaningful work
   (not just returning arbitrary chunks).
4. **Answer groundedness** — semantic similarity between the generated answer
   and its cited context, as a lightweight hallucination check (a low score
   suggests the LLM strayed from the retrieved context).

> **Before running**: update `qa_dataset.json` with real questions about a
> document you've actually uploaded through the app, and fill in `BACKEND_URL`,
> `EMAIL`, and `PASSWORD` below.

In [ ]:
import json
import requests
import numpy as np
import matplotlib.pyplot as plt

BACKEND_URL = "http://localhost:8000"   # or your deployed Render URL
EMAIL = "admin@example.com"
PASSWORD = "change_this_admin_password"  # must match your .env ADMIN_PASSWORD
DOCUMENT_ID = None  # optionally restrict eval to a single uploaded document's id

with open("qa_dataset.json") as f:
    qa_dataset = json.load(f)

len(qa_dataset), qa_dataset[0]

## 1. Authenticate

In [ ]:
resp = requests.post(f"{BACKEND_URL}/auth/login", json={"email": EMAIL, "password": PASSWORD})
resp.raise_for_status()
token = resp.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"}
print("Authenticated OK")

## 2. Run every question through the pipeline

Each call hits `/chat/ask`, which runs the *full* pipeline: embed query →
dense (vector) retrieval → sparse (BM25) retrieval → merge → cross-encoder
rerank → LLM generation. We capture the full retrieval trace returned in
`sources`, not just the final answer.

In [ ]:
results = []
for item in qa_dataset:
    r = requests.post(
        f"{BACKEND_URL}/chat/ask",
        json={"question": item["question"], "document_id": DOCUMENT_ID},
        headers=headers,
    )
    r.raise_for_status()
    data = r.json()
    results.append({**item, "answer": data["answer"], "sources": data["sources"]})

for res in results:
    print("Q:", res["question"])
    print("A:", res["answer"][:200], "...\n")

## 3. Retrieval Hit-Rate & MRR

A "hit" = at least one of the expected keywords appears in a retrieved
chunk's text. Rank of the *first* hit determines the reciprocal rank for MRR.

In [ ]:
def first_hit_rank(expected_keywords, sources):
    for rank, s in enumerate(sources, start=1):
        text = (s.get("preview") or "").lower()
        if any(kw.lower() in text for kw in expected_keywords):
            return rank
    return None

hit_flags, reciprocal_ranks = [], []
for res in results:
    rank = first_hit_rank(res["expected_keywords"], res["sources"])
    hit_flags.append(rank is not None)
    reciprocal_ranks.append(1.0 / rank if rank else 0.0)

hit_rate = np.mean(hit_flags)
mrr = np.mean(reciprocal_ranks)
print(f"Hit-Rate@{len(results[0]['sources']) if results else 0}: {hit_rate:.2%}")
print(f"MRR: {mrr:.3f}")

## 4. Score breakdown per question (BM25 vs Vector vs Rerank)

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4), squeeze=False)

for i, res in enumerate(results):
    ax = axes[0][i]
    labels = [f"#{j+1}" for j in range(len(res["sources"]))]
    bm25 = [s.get("bm25_score") or 0 for s in res["sources"]]
    vec = [(s.get("vector_score") or 0) * 10 for s in res["sources"]]  # scaled for visibility
    rrk = [s.get("rerank_score") or 0 for s in res["sources"]]

    x = np.arange(len(labels))
    width = 0.25
    ax.bar(x - width, bm25, width, label="BM25")
    ax.bar(x, vec, width, label="Vector x10")
    ax.bar(x + width, rrk, width, label="Rerank")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title(res["question"][:40] + "...", fontsize=9)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig("score_breakdown.png", dpi=120)
plt.show()

## 5. Answer Groundedness (lightweight hallucination check)

Embeds the generated answer and its retrieved context separately, then
computes cosine similarity. A consistently low score across questions would
suggest the LLM is drifting away from the retrieved context rather than
staying grounded in it.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

groundedness_scores = []
for res in results:
    context = " ".join(s["preview"] for s in res["sources"])
    if not context.strip():
        groundedness_scores.append(0.0)
        continue
    emb_answer = embed_model.encode(res["answer"])
    emb_context = embed_model.encode(context)
    groundedness_scores.append(cosine(emb_answer, emb_context))

for res, score in zip(results, groundedness_scores):
    print(f"{score:.3f}  {res['question'][:60]}")

print("\nMean groundedness:", np.mean(groundedness_scores))

## 6. Summary

Prints a compact table you can paste into a README/report to summarize
pipeline quality on this sample set.

In [ ]:
print(f"{'Metric':<30}{'Value'}")
print("-" * 40)
print(f"{'Hit-Rate@k':<30}{hit_rate:.2%}")
print(f"{'MRR':<30}{mrr:.3f}")
print(f"{'Mean Groundedness':<30}{np.mean(groundedness_scores):.3f}")